# Approach 2: Biomedical Hybrid Retrieval + QLoRA Fine-Tuning

This notebook demonstrates the end-to-end pipeline for Approach 2:
1. **Biomedical Retrieval & Cache**: Chunk-weighted PubMedBERT dense embeddings combined with lexical TF-IDF and modality matching.
2. **ChatML Preprocessing**: Qwen2.5-3B-Instruct chat template formatting with whole-demonstration budget pruning.
3. **Target-Only Completion Masking**: Prompt tokens are ignored and loss applies only to assistant report tokens.
4. **QLoRA Fine-Tuning**: 4-bit NF4 training with rank-16 LoRA adapters and TRL.
5. **Adapter Inference**: Greedy continuation decoding.
6. **RES Evaluation**: Clinically weighted evaluation using the canonical RES implementation.

In [ ]:
from pathlib import Path
import torch

from hybrid_retrieval_qlora.config import (
    DEFAULT_CACHE_DIR,
    clinical_data_root,
)
from hybrid_retrieval_qlora.preprocess.load_data import (
    load_cases,
    create_deterministic_splits,
    filter_training_pool,
)
from hybrid_retrieval_qlora.retrieval.cache_manifest import EmbeddingCacheManager, EmbeddingSpec
from hybrid_retrieval_qlora.retrieval.pubmedbert_embeddings import PubMedBERTEmbedder
from hybrid_retrieval_qlora.retrieval.hybrid_retriever import HybridRetriever
from hybrid_retrieval_qlora.preprocess.completion_mask import get_qwen_tokenizer
from hybrid_retrieval_qlora.training.qlora_config import QLoRAConfig
from hybrid_retrieval_qlora.training.sft_dataset import prepare_sft_records, to_hf_dataset
from hybrid_retrieval_qlora.training.train import train_qlora
from hybrid_retrieval_qlora.inference.generate_reports import generate_reports
from hybrid_retrieval_qlora.training.evaluate import evaluate_reports


## 1. Load Data & Deterministic Splits

Partition clinical cases into train, validation, and test splits with strict leakage protection.

In [ ]:
data_dir = clinical_data_root()
train_csv = data_dir / "train.csv"
print(f"Data directory: {data_dir}")

if train_csv.is_file():
    cases = load_cases(train_csv, require_report=True)
    print(f"Loaded {len(cases)} clinical cases from {train_csv.name}.")
else:
    print("Private dataset not found. Using synthetic fixture cases for demonstration.")
    from tests.test_qlora_training import _create_sample_cases
    cases = _create_sample_cases()

manifest, train_cases, val_cases, test_cases = create_deterministic_splits(cases, seed=42)
training_pool = filter_training_pool(cases, manifest)
print(f"Partitions: {len(train_cases)} train, {len(val_cases)} val, {len(test_cases)} test.")
print(f"Candidate demonstration pool size: {len(training_pool)} cases.")

## 2. Initialize Biomedical Hybrid Retriever & Cache

Setup PubMedBERT embedding cache and build the hybrid retrieval index on training-only candidates.

In [ ]:
spec = EmbeddingSpec()
cache_mgr = EmbeddingCacheManager(DEFAULT_CACHE_DIR, spec=spec)
embedder = PubMedBERTEmbedder(
    model_name=spec.model,
    revision=spec.revision,
    chunk_size=spec.chunk_size,
    chunk_overlap=spec.overlap,
)
retriever = HybridRetriever(training_pool, manifest=manifest, top_k=5)
print(f"Retriever configured with weights: {retriever.weights}")

## 3. Prepare SFT Dataset with Target-Only Completion Masking

Format training and validation cases with Hugging Face ChatML template and compute loss masks.

In [ ]:
config = QLoRAConfig()
try:
    tokenizer = get_qwen_tokenizer(config.model_name, config.model_revision)
    print(f"Loaded Qwen tokenizer: {config.model_name}")
except Exception as exc:
    print(f"AutoTokenizer offline fallback: {exc}")
    from hybrid_retrieval_qlora.preprocess.completion_mask import MockQwenTokenizer
    tokenizer = MockQwenTokenizer()

train_records, train_usages, train_formatted = prepare_sft_records(
    train_cases,
    tokenizer,
    retriever=retriever,
    cache_manager=cache_mgr,
    embedder=embedder,
    supervised=True,
    top_k=5,
    max_input_tokens=config.max_input_tokens,
    max_seq_length=config.max_seq_length,
)
val_records, val_usages, val_formatted = prepare_sft_records(
    val_cases,
    tokenizer,
    retriever=retriever,
    cache_manager=cache_mgr,
    embedder=embedder,
    supervised=True,
    top_k=5,
    max_input_tokens=config.max_input_tokens,
    max_seq_length=config.max_seq_length,
)

train_dataset = to_hf_dataset(train_records)
val_dataset = to_hf_dataset(val_records)
print(f"Built SFT train dataset ({len(train_records)} rows) and val dataset ({len(val_records)} rows).")
first_rec = train_records[0]
print(f"First record: {len(first_rec.input_ids)} tokens | supervised completion tokens: {first_rec.completion_mask.count(1)}")

## 4. QLoRA 4-bit NF4 Training

Fine-tune Qwen2.5-3B-Instruct using PEFT LoRA adapters and TRL SFTTrainer.

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
train_result = None
if torch.cuda.is_available():
    train_result = train_qlora(
        config,
        train_dataset,
        val_dataset,
        tokenizer,
    )
    print(f"Training finished! Adapter saved to: {train_result['adapter_dir']}")
else:
    print("GPU kernel required for 4-bit NF4 fine-tuning. Skipping local training.")

## 5. Report Generation & RES Evaluation

Generate adapter completions on evaluation cases and score with canonical RES metric.

In [ ]:
eval_cases = test_cases if test_cases else val_cases
print(f"Ready for evaluation on {len(eval_cases)} cases.")
if train_result is None:
    print("Skipping inference and RES evaluation because QLoRA training did not run.")
else:
    gen_results = generate_reports(
        train_result["model"],
        tokenizer,
        eval_cases,
        retriever=retriever,
        cache_manager=cache_mgr,
        embedder=embedder,
        top_k=5,
        max_new_tokens=config.max_new_tokens,
        output_path=Path(train_result["run_dir"]) / "test_reports.csv",
    )
    scores, summary = evaluate_reports(
        eval_cases,
        gen_results,
        output_path=Path(train_result["run_dir"]) / "test_scores.csv",
    )
    print(summary)